In [2]:
import pandas as pd
import numpy as np

data = {
    "customer_id": [1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 10, 11, 12, 13, 14],
    "age": [25, 30, 35, 40, 45, 25, 30, 35, 40, 45, 50, -5, 29, 33, 41],
    "income": [
        50000, 60000, 55000, 58000, 62000,
        50000, 60000, 55000, 58000, 62000,
        999999999, 45000, np.nan, np.nan, np.nan
    ],
    "purchases": [3, 4, 2, 5, 6, 3, 4, 2, 5, 6, 10, 2, "ABC", 4, 5]
}

df = pd.DataFrame(data)

print("Dataset:")
print(df)

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("Duplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

Dataset:
    customer_id  age       income purchases
0             1   25      50000.0         3
1             2   30      60000.0         4
2             3   35      55000.0         2
3             4   40      58000.0         5
4             5   45      62000.0         6
5             1   25      50000.0         3
6             2   30      60000.0         4
7             3   35      55000.0         2
8             4   40      58000.0         5
9             5   45      62000.0         6
10           10   50  999999999.0        10
11           11   -5      45000.0         2
12           12   29          NaN       ABC
13           13   33          NaN         4
14           14   41          NaN         5

Dataset Shape:
(15, 4)

Missing Values:
customer_id    0
age            0
income         3
purchases      0
dtype: int64
Duplicate Rows:
5

Data Types:
customer_id      int64
age              int64
income         float64
purchases       object
dtype: object


In [3]:
def quality_report(dataframe):
    report = {}

    # 1. Dataset shape
    report["rows"] = dataframe.shape[0]
    report["columns"] = dataframe.shape[1]

    # 2. Missing values
    report["missing_values"] = dataframe.isnull().sum().to_dict()

    # 3. Duplicate rows
    report["duplicate_rows"] = dataframe.duplicated().sum()

    # 4. Data types
    report["data_types"] = dataframe.dtypes.astype(str).to_dict()

    # 5. Invalid age check
    invalid_age_count = dataframe[dataframe["age"] < 0].shape[0]
    report["invalid_age_count"] = invalid_age_count

    # 6. Invalid purchases check
    purchases_numeric = pd.to_numeric(dataframe["purchases"], errors="coerce")

    invalid_purchases_count = (
        purchases_numeric.isnull().sum()
        - dataframe["purchases"].isnull().sum()
    )

    report["invalid_purchases_count"] = invalid_purchases_count

    # 7. Income outlier check using IQR
    income_clean = dataframe["income"].dropna()

    Q1 = income_clean.quantile(0.25)
    Q3 = income_clean.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    income_outliers_count = dataframe[
        (dataframe["income"] < lower_bound) |
        (dataframe["income"] > upper_bound)
    ].shape[0]

    report["income_outliers_count"] = income_outliers_count

    # 8. Total issues
    total_issues = (
        dataframe.isnull().sum().sum()
        + dataframe.duplicated().sum()
        + invalid_age_count
        + invalid_purchases_count
        + income_outliers_count
    )

    report["total_issues"] = int(total_issues)

    return report

In [4]:
before_report = quality_report(df)

print("\nQuality Report Before Cleaning:")
print(before_report)


Quality Report Before Cleaning:
{'rows': 15, 'columns': 4, 'missing_values': {'customer_id': 0, 'age': 0, 'income': 3, 'purchases': 0}, 'duplicate_rows': np.int64(5), 'data_types': {'customer_id': 'int64', 'age': 'int64', 'income': 'float64', 'purchases': 'object'}, 'invalid_age_count': 1, 'invalid_purchases_count': np.int64(1), 'income_outliers_count': 1, 'total_issues': 11}


In [6]:
def clean_data(dataframe):
    df_clean = dataframe.copy()

    # 1. Remove duplicate rows
    df_clean = df_clean.drop_duplicates()

    # 2. Remove invalid age values
    df_clean = df_clean[df_clean["age"] >= 0]

    # 3. Convert purchases column to numeric
    df_clean["purchases"] = pd.to_numeric(df_clean["purchases"], errors="coerce")

    # 4. Drop rows where purchases became NaN
    df_clean = df_clean.dropna(subset=["purchases"])

    # 5. Handle income outlier using IQR clipping
    Q1 = df_clean["income"].quantile(0.25)
    Q3 = df_clean["income"].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_clean["income"] = df_clean["income"].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    # 6. Fill missing income using median
    df_clean["income"] = df_clean["income"].fillna(df_clean["income"].median())

    return df_clean

In [7]:
cleaned_df = clean_data(df)

print("\nCleaned Dataset:")
print(cleaned_df)

after_report = quality_report(cleaned_df)

print("\nQuality Report After Cleaning:")
print(after_report)


Cleaned Dataset:
    customer_id  age   income  purchases
0             1   25  50000.0        3.0
1             2   30  60000.0        4.0
2             3   35  55000.0        2.0
3             4   40  58000.0        5.0
4             5   45  62000.0        6.0
10           10   50  70125.0       10.0
13           13   33  59000.0        4.0
14           14   41  59000.0        5.0

Quality Report After Cleaning:
{'rows': 8, 'columns': 4, 'missing_values': {'customer_id': 0, 'age': 0, 'income': 0, 'purchases': 0}, 'duplicate_rows': np.int64(0), 'data_types': {'customer_id': 'int64', 'age': 'int64', 'income': 'float64', 'purchases': 'float64'}, 'invalid_age_count': 0, 'invalid_purchases_count': np.int64(0), 'income_outliers_count': 2, 'total_issues': 2}
